# JevGAG — A100 setup (E3 / E8)

**Jev-Grounded Adaptive Temporal Granularity for Causal Video Generation.**

Repo: https://github.com/j1s4nn/JevGAG  ·  Site: https://j1s4nn.github.io/JevGAG/

**Runtime → Change runtime type → A100 GPU** before running.

This notebook sets up the environment (GPU check, deps, checkpoints) and confirms readiness.
It spends **no** A100 compute units until the training/generation cells are run.

In [ ]:
import subprocess, sys
print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout)
import torch
print("torch", torch.__version__, "cuda", torch.cuda.is_available())
assert torch.cuda.is_available(), "Enable a GPU runtime first."
name = torch.cuda.get_device_name(0)
print("GPU:", name)
assert "A100" in name, f"This notebook expects A100; got {name}"
print("OK: A100 ready")

In [ ]:
packages = ["einops","transformers","sentencepiece","ftfy","regex","imageio","imageio-ffmpeg",
            "tqdm","torchvision","fvcore","omegaconf","attrs","pyyaml","iopath","termcolor",
            "pynvml","pandas","loguru","safetensors","cloudpickle","dill","matplotlib",
            "packaging","opencv-python-headless","kaggle"]
subprocess.run([sys.executable,"-m","pip","install","-q"]+packages, check=True)
print("deps installed")

In [ ]:
# Kaggle credentials (for the checkpoints). Paste when prompted, or set via Colab Secrets.
import os
os.environ.setdefault("KAGGLE_USERNAME", "ajjisan")
try:
    from google.colab import userdata
    os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
except Exception:
    import getpass
    if not os.environ.get("KAGGLE_KEY"):
        os.environ["KAGGLE_KEY"] = getpass.getpass("KAGGLE_KEY: ")
print("kaggle user:", os.environ["KAGGLE_USERNAME"], "key set:", bool(os.environ.get("KAGGLE_KEY")))

In [ ]:
# Clone this repo and fetch the prior A100 reference scripts (reused, read-only).
import os
if not os.path.isdir("/content/JevGAG"):
    subprocess.run(["git","clone","--depth","1","https://github.com/j1s4nn/JevGAG.git","/content/JevGAG"], check=True)
print(sorted(os.listdir("/content/JevGAG/colab")))

In [ ]:
# Download the causal-rCM checkpoints (fine c1-1 s2/s4, coarse c3-3 s2) from Kaggle.
import subprocess
os.makedirs("/content/inputs", exist_ok=True)
subprocess.run(["kaggle","datasets","download","-d","ajjisan/causal-rcm-ckpts",
                "-p","/content/inputs","--unzip","-q"], check=True)
for root,_,files in os.walk("/content/inputs"):
    for f in files:
        if f.endswith(".pt"):
            print(os.path.join(root,f))
print("checkpoints ready")

In [ ]:
# Optional: hand CLI control to the JevGAG driver (colab-ssh + cloudflared tunnel).
# After this cell prints an SSH command, the driver can run everything on this VM from the terminal.
!pip -q install colab_ssh --upgrade
from colab_ssh import launch_ssh_cloudflared
launch_ssh_cloudflared(password="jevGAG")

## Ready
Environment is set up. E3 (matched online factorial) and E8 (stability-gated recalibration)
run cells are added by the JevGAG driver when the main task starts.